# Case Study 1: Evolution of Netflix's Content Catalog

**Course:** ITCS 5123 — Information Visualization
**Author:** Micah Platte
**Dataset:** Netflix Movies and Series (Kaggle)
**Platform:** Python, Pandas, Plotly

## Introduction
This case study examines how Netflix's content catalog has changed over time by analyzing the distribution of movies, TV shows, genres, and content ratings. Using the Netflix Movies and Series dataset from Kaggle, three visualizations are developed to identify patterns in content additions and catalog composition. The study investigates changes in the number of titles added each year, the representation of popular genres over time, and differences in content ratings between movies and television shows. These visualizations provide an overview of how Netflix's catalog is structured and how its composition has evolved.

## Research Background
Ali et al. (2025), Data-Driven Insights: Machine Learning Approaches for Netflix Content Analysis and Visualization.

This paper explores Netflix's content library using Python and Tableau to create several types of visualizations, including pie charts, line graphs, histograms, and treemaps. The researchers use these visualizations to compare content across countries, analyze movie and TV show ratings, identify common genres, and track how Netflix's catalog has changed over time. One major finding was that the United States and India contributed the largest portions of content, while TV-MA was one of the most common ratings. A line graph also showed a major increase in Netflix content additions between 2014 and 2020.

This study is especially relevant to our project because it explores many of the same questions we want to answer about Netflix's catalog. Its use of line graphs to show changes over time and charts to compare countries and genres provides a useful starting point for our visualization designs. We can build on these ideas by adding interactive filters for year, genre, content type, and rating, allowing users to explore patterns and compare different parts of Netflix's catalog more easily.

We aim to expand on the paper's descriptive analysis methods by using interactive Plotly visualizations that allow users to inspect content trends and compare categories.

## Data
We are using a cleaned version of the Netflix Movies and Series dataset sourced from Kaggle.
https://www.kaggle.com/datasets/debayank2024/netflix-movies-and-series

All data cleaning was performed in src/clean-data.py as a means to ensure visualizations are accurate and relevant. Changes made are described in comments within the same file and were made to address any accuracy or aggregation concerns.

For this case study we will focus on the following fields: show_id, type, date_added, year_added, list_in, and rating

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path

project_root = Path.cwd()

if not (project_root / "data" / "netflix_cleaned.csv").exists():
    project_root = project_root.parent

df = pd.read_csv(project_root / "data" / "netflix_cleaned.csv")

df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

## Visualization 1: Movies vs. TV Shows Over Time
Research question:
How has the number of movies and TV shows added to Netflix changed over time?

**Visualization Scope:** Although the dataset contains records of titles added before 2016, these years have relatively few entries compared with later periods. To make the trends easier to interpret, this visualization focuses on titles added from 2016 onward. Earlier records are retained in the dataset but excluded from this particular view. This allows for a clearer comparison of movie and TV show additions during a period with greater data representation.

In [ ]:
yearly_counts = (
    df.dropna(subset=["year_added"])
      .query("year_added >= 2016")
      .groupby(["year_added", "type"])
      .size()
      .reset_index(name="count")
)

print(
    yearly_counts.pivot(
        index="year_added",
        columns="type",
        values="count"
    )
)

fig = px.line(
    yearly_counts,
    x="year_added",
    y="count",
    color="type",
    markers=True,
    title="Netflix Movies and TV Shows Added (2016–2021)",
    labels={
        "year_added": "Year Added",
        "count": "Number of Titles",
        "type": "Content Type"
    }
)

fig.show()

type        Movie  TV Show
year_added                
2016.0        253      176
2017.0        839      349
2018.0       1237      412
2019.0       1423      592
2020.0       1284      595
2021.0        993      505


Write a paragraph answering:
- Which content type is more common?
- When do the largest increases occur?
- Are movies and TV shows following similar trends?
- Do your findings agree with Ali et al.'s observations about content additions?
Avoid stating that Netflix's total catalog shrank merely because annual additions decreased. This chart measures additions, not removals or the total catalog size at each point in time.

Based on the graph, we can clearly see that overall TV Shows have a higher number of entries in the catalog than movies, this is consistent across all years from 2016-2021. The largest increases in both categories occurred in 2019, with a slow decrease in years after, from this we can note that the number of TV Shows and Movies being added each year follow similar trends within their respective categories. 

## Visualization 2: Genre Trends Over Time
Research question
How has the representation of popular Netflix genres changed over time?

**Visualization Scope:** Although the dataset contains records of titles added before 2016, these years have relatively few entries compared with later periods. To make the trends easier to interpret, this visualization focuses on titles added from 2016 onward. Earlier records are retained in the dataset but excluded from this particular view. This allows for a clearer comparison of movie and TV show additions during a period with greater data representation.

In [ ]:
# Filter to titles added from 2016 onward
genre_df = df.dropna(subset=["year_added"]).copy()
genre_df = genre_df[genre_df["year_added"] >= 2016]

# Convert years to integers
genre_df["year_added"] = genre_df["year_added"].astype(int)

# Separate multiple genres into individual rows
genre_df["genre"] = genre_df["listed_in"].str.split(",")
genre_df = genre_df.explode("genre")
genre_df["genre"] = genre_df["genre"].str.strip()

# Identify the 10 most common genres
top_genres = genre_df["genre"].value_counts().head(10).index

# Keep only the top genres
genre_df = genre_df[genre_df["genre"].isin(top_genres)]

# Count titles by genre and year
genre_counts = (
    genre_df.groupby(["genre", "year_added"])
    .size()
    .unstack(fill_value=0)
)

# Create heatmap
fig = px.imshow(
    genre_counts,
    labels={
        "x": "Year Added",
        "y": "Genre",
        "color": "Number of Titles"
    },
    title="Netflix Genre Trends Over Time (2016–2021)",
    aspect="auto",
    text_auto=True,
    color_continuous_scale="Blues"
)

fig.update_layout(
    xaxis_title="Year Added",
    yaxis_title="Genre",
    height=550
)

fig.show()

What to analyze
- Which genres appear most frequently?
- Are some genres consistently more represented than others?
- Do certain genres become more prominent in later years?
- Are changes in genre counts similar to overall content additions?
Mention that Netflix's genre labels can overlap. One title may contribute to several genre categories.

## 6. Visualization 3: Content Ratings by Type
Research question
How do content-rating distributions differ between Netflix movies and TV shows?

In [ ]:
rating_counts = (
    df.groupby(["type", "rating"])
    .size()
    .reset_index(name="count")
)

rating_counts["percentage"] = (
    rating_counts["count"] /
    rating_counts.groupby("type")["count"].transform("sum")
    * 100
)

fig = px.bar(
    rating_counts,
    x="type",
    y="percentage",
    color="rating",
    title="Netflix Content Ratings by Type",
    labels={
        "type": "Content Type",
        "percentage": "Percentage of Titles",
        "rating": "Content Rating"
    }
)

fig.update_layout(barmode="stack")
fig.show()

What to analyze
- Which ratings are most common among movies?
- Which ratings are most common among TV shows?
- Does mature-rated content represent a large proportion of either category?
- How do these results compare with Ali et al.'s findings?
Since movies and TV shows have different total counts, percentages make the comparison more meaningful.

After completing all three visualizations, write approximately 200–300 words connecting the results.

This section should explain what you learned from considering the charts together.

For example:
- Do years with more content additions also show increases in major genres?
- Does the dominance of movies influence the overall genre distribution?
- Do content ratings reveal differences that are not obvious from total title counts?
- How do your observations compare with the original paper?
You should explicitly reference each figure:

"Figure 1 demonstrates..."

"Figure 2 provides additional evidence..."

"Figure 3 reveals..."

Use your actual outputs to support these statements rather than writing conclusions before running the visualizations.

Markdown cell
Your limitations section should acknowledge the following:
- The dataset is a snapshot and may not reflect Netflix's current catalog.
- Recorded additions do not establish how many titles remained available in any particular year.
- Genre categories overlap.
- Missing dates and other metadata can affect counts.
- Content ratings represent age classifications, not viewer satisfaction.
- The dataset does not include streaming statistics or viewership.
Finish with a short conclusion summarizing the most important findings and how the visualizations helped answer your research questions.

Because LLM integration is explicitly part of the assignment, document how you used ChatGPT during the project.

## References:
Netflix Movies and Series dataset sourced from Kaggle.
https://www.kaggle.com/datasets/debayank2024/netflix-movies-and-series

Ali et al. (2025), Data-Driven Insights: Machine Learning Approaches for Netflix Content Analysis and Visualization.